# 27. 文本、日期与特征处理

<!-- module-learning-arc:start -->
> **Pandas 模块主线｜第 6 / 10 步：修正类型并建立干净字段**
>
> **持续应用背景：** 搭建电商履约异常追踪台：把订单、客户、商品和履约信息整理成安全合并的事实表，再生成趋势指标和异常工单。
>
> **承接上一阶段：** 数据质量检查与清洗  →  **本章任务：** 文本、日期与特征处理  →  **下一步：** 数据读取与保存
>
> **大作业连接：** 本章练习将成为《电商履约异常追踪台》的一部分，最终需要从多表质量审计走到订单粒度事实表、窗口趋势和可复核异常工单。
<!-- module-learning-arc:end -->


## 本章场景

处理好手里的数据，是后面所有统计和建模能跑起来的前提。很多原始字段并不"干净"：姓名前后带着空格、电话号混着横线、日期存成文本，想按月份分组或算两天的间隔，就得先把字段的"样子"整理对。这一节先把三种最常用的整理手段——文本清洗、日期转换、特征构造——讲成概念，后面再用示例和练习把它们串起来。- Pandas字符串方法通过str访问器调用。- 日期必须先转换为datetime才能进行时间运算。- 特征应由业务问题驱动，并避免使用未来信息。



## 本章目标

学完本章，你将能够：

- **理解**：理解文本清洗、日期解析与特征构造。
- **操作**：能把文本/日期列转成可用特征（分类、数值、时间）。
- **迁移**：能从不规整的原始字段中构造出可用于分析的特征。


## 27.1 核心概念

**背景引入**：处理好手里的数据，是后面所有统计和建模能跑起来的前提。很多原始字段并不"干净"：姓名前后带着空格、电话号混着横线、日期存成文本，想按月份分组或算两天的间隔，就得先把字段的"样子"整理对。这一节先把三种最常用的整理手段——文本清洗、日期转换、特征构造——讲成概念，后面再用示例和练习把它们串起来。- Pandas字符串方法通过str访问器调用。- 日期必须先转换为datetime才能进行时间运算。- 特征应由业务问题驱动，并避免使用未来信息。

> **直观类比**：日期存在文本里，就像写在“纸条”上，只能读不能算；转成 datetime 就像把纸条换成“日历上具体的一天”，才能比大小、数间隔、按月归类。


## 27.2 方法分类速查

先用这张表建立本章的方法地图；每一行后面都有对应的独立示例或练习。

| 类别 | 常用方法或写法 | 主要用途 | 需要特别注意 |
| --- | --- | --- | --- |
| Series.str.strip() | `pd.Series()`、`names.str.strip()` | str访问器可以批量调用字符串方法。 | 直接对object列使用日期运算 |
| Series.str.replace() | `pd.Series()`、`phones.str.replace()` | 正则替换适合清理格式不一致的文本。 | 正则提取失败后不检查缺失 |
| to_datetime() | `pd.Series()`、`pd.to_datetime()`、`parsed.isna()`、`.sum()` | 无效日期可以转为NaT，但转换后必须检查缺失。 | 使用结果变量构造导致数据泄漏的特征 |
| dt 访问器 | `pd.to_datetime()`、`pd.Series()`、`dates.dt.month.tolist()`、`dates.dt.day_name()` | 日期解析后通过dt访问器提取月份、星期等字段。 | 直接对object列使用日期运算 |
| cut() | `pd.Series()`、`pd.cut()`、`float()` | cut把连续数值转换为有序区间类别。 | 正则提取失败后不检查缺失 |


## 27.3 示例 1：文本标准化

**背景引入**：客户表里的姓名前后带着空格、大小写五花八门，邮箱域名也想单独统计——这种"看着差不多、实际对不上"的文本，直接拿去分组或匹配必然漏人。先把文本的"样子"统一，才能安心做后续分析。

**讲解**：用 `str` 访问器批量调用字符串方法，一口气完成去空格、去中间空格、统一小写和正则提取。

- `str.strip()` 去掉名字两侧空格，`.str.replace(" ", "", regex=False)` 再去掉中间空格；
- `str.lower()` 把邮箱统一成小写，避免 "A@EXAMPLE.COM" 和 "a@example.com" 被当成两个人；
- `str.extract(r"@(.+)$", expand=False)` 用正则提出 @ 后的域名，`expand=False` 返回 Series；
- **口诀**：str 前面加一点，"统一与提取"一串搞定，文本先洗白再分析。


In [ ]:
import pandas as pd

customers = pd.DataFrame(
    {
        "name": [" 张三 ", "LI SI", "王 五"],
        "email": ["A@EXAMPLE.COM", "li@test.cn", "wang@example.com"],
    }
)
customers["name_clean"] = (
    customers["name"].str.strip().str.replace(" ", "", regex=False)
)
customers["email_clean"] = customers["email"].str.strip().str.lower()
customers["domain"] = customers["email_clean"].str.extract(
    r"@(.+)$", expand=False
)
print(customers)


## 27.4 示例 2：日期解析与拆解

**背景引入**：日期明明长得很像，却存成了文本（字符串）。想在订单里按月份分组、数一数星期几卖得最好，直接对字符串做时间运算是行不通的，而且里面还混着一行 "invalid" 的脏日期。先把日期转成真正的日期类型，问题就解开一大半。

**讲解**：`pd.to_datetime(..., errors="coerce")` 把日期列转成 datetime，无效值变成 NaT；再用 `dt` 访问器拆出月、星期等字段。

- `errors="coerce"` 让 "invalid" 这类坏日期变成 NaT 而不报错中断；
- `dt.to_period("M")` 提取到"月份"这一粒度，`.astype("string")` 转成方便显示的文本；
- `dt.day_name()` 直接取出星期名，适合做星期维度的销售分析；
- **口诀**：日期先 to_datetime，坏值 coerce 变 NaT，要月要星期全靠 dt 拆。


In [ ]:
orders = pd.DataFrame(
    {
        "order_date": ["2026-01-05", "2026-02-18", "invalid", "2026-03-22"],
        "amount": [320, 880, 460, 1250],
    }
)
orders["order_date"] = pd.to_datetime(orders["order_date"], errors="coerce")
orders["month"] = orders["order_date"].dt.to_period("M").astype("string")
orders["weekday"] = orders["order_date"].dt.day_name()
print(orders)


## 27.5 示例 3：特征构造

**背景引入**：原始字段往往不够用——运营想知道"这张单距离今天多久了"、"金额算普通还是大额"，这些都得从已有字段"造"出新字段。造特征最怕口径含糊，必须让每个新字段能从原始数据一步步复算出来。

**讲解**：基于参考日和订单日期做时间差得到"距今天数"，再用 `cut` 把连续金额切成有序等级。

- `reference_date - orders["order_date"]` 做 datetime 减法，`.dt.days` 取出间隔天数；
- `pd.cut(amount, bins=[0,500,1000,inf], labels=[...])` 把金额按区间切成有序类别；
- 特征口径明确：`days_ago` 从参考日复算、`amount_level` 由区间定义，别人能验证；
- **口诀**：新字段要能复算，差值取天、cut 分等级，口径说清不背锅。


In [ ]:
reference_date = pd.Timestamp("2026-04-01")
orders["days_ago"] = (reference_date - orders["order_date"]).dt.days
orders["amount_level"] = pd.cut(
    orders["amount"],
    bins=[0, 500, 1000, float("inf")],
    labels=["普通", "重点", "大额"],
)
print(orders)


## 27.6 核心操作独立示例

下面每个代码单元格只演示一个核心方法、函数或语法操作。请先阅读方法名称和任务说明，再单独运行当前单元格；示例尽量自带最小输入，不要求依赖前一个单元格留下的变量。


In [ ]:
# Series.str.strip()
# str访问器可以批量调用字符串方法。
import pandas as pd

names = pd.Series([" 张三 ", " 李四"])
print(names.str.strip())


In [ ]:
# Series.str.replace()
# 正则替换适合清理格式不一致的文本。
import pandas as pd

phones = pd.Series(["138-0000-1234", "139 0000 5678"])
print(phones.str.replace(r"\D", "", regex=True))


In [ ]:
# to_datetime()
# 无效日期可以转为NaT，但转换后必须检查缺失。
import pandas as pd

dates = pd.Series(["2026-01-05", "invalid"])
parsed = pd.to_datetime(dates, errors="coerce")
print(parsed)
print("无效日期数:", parsed.isna().sum())


In [ ]:
# dt 访问器
# 日期解析后通过dt访问器提取月份、星期等字段。
import pandas as pd

dates = pd.to_datetime(pd.Series(["2026-01-05", "2026-02-18"]))
print(dates.dt.month.tolist())
print(dates.dt.day_name().tolist())


In [ ]:
# cut()
# cut把连续数值转换为有序区间类别。
import pandas as pd

amount = pd.Series([120, 580, 1280])
level = pd.cut(
    amount, bins=[0, 500, 1000, float("inf")], labels=["普通", "重点", "大额"]
)
print(level)


**练一练 23.6**：小订单表的三步处理下面是一张小订单表，只有 3 个订单。请分三步完成：先把客户名两端的空格去掉，再把下单日期转成真正的日期并取出月份，最后按金额把订单分成"普通"（金额 < 500）与"大额"（金额 ≥ 500）两级。请在下面代码单元格的填空处补全代码，然后运行。先看输出是否符合预期，再核对末尾的自检是否通过；需要提示时，回看上面的 `Series.str.strip()`、`to_datetime()`、`dt 访问器` 和 `cut()` 四个独立示例。


In [ ]:
# 请在下方填写代码
import pandas as pd

# 第 1 步：去掉客户名两端的空格
# TODO: orders["customer_clean"] = ...
# 第 2 步：把 date 转成日期类型，并取出月份
# TODO: orders["date"] = pd.to_datetime(...)
# TODO: orders["month"] = orders["date"].dt.month
# 第 3 步：按金额分成 "普通"（<500）与 "大额"（>=500）两级
# TODO: orders["level"] = pd.cut(...)


In [ ]:
import pandas as pd

orders = pd.DataFrame(
    {
        "customer": [" 张三 ", " 李四 ", " 王五 "],
        "date": ["2026-01-12", "2026-02-03", "2026-03-25"],
        "amount": [80, 1200, 300],
    }
)

# 第 1 步：去掉客户名两端的空格
orders["customer_clean"] = orders["customer"].str.strip()

# 第 2 步：把 date 转成日期类型，并取出月份
orders["date"] = pd.to_datetime(orders["date"], errors="coerce")
orders["month"] = orders["date"].dt.month

# 第 3 步：按金额分成 "普通"（<500）与 "大额"（>=500）两级
orders["level"] = pd.cut(
    orders["amount"],
    bins=[0, 500, float("inf")],
    labels=["普通", "大额"],
)

print(orders)


## 27.7 公开大型数据实战

下面使用 UCI Machine Learning Repository 的 Online Retail 公开数据集。原始数据包含 541,909 条英国在线零售交易，本课程使用固定随机种子抽取的 200,000 行子集。分析时在完整子集上计算，只展示摘要或少量样本。


In [ ]:
import numpy as np
import pandas as pd

# UCI Machine Learning Repository: Online Retail
# 原始数据 541,909 行；课程使用固定随机种子抽取的 200,000 行子集。
data_url = "/datasets/uci_online_retail_200k.csv"
large_orders = pd.read_csv(
    data_url,
    parse_dates=["InvoiceDate"],
    dtype={
        "InvoiceNo": "string",
        "StockCode": "string",
        "Description": "string",
        "Country": "category",
    },
).rename(
    columns={
        "InvoiceNo": "order_id",
        "StockCode": "stock_code",
        "Description": "description",
        "Quantity": "quantity",
        "InvoiceDate": "order_time",
        "UnitPrice": "unit_price",
        "CustomerID": "customer_id",
        "Country": "country",
    }
)
large_orders["sales"] = (
    large_orders["quantity"] * large_orders["unit_price"]
).round(2)
large_orders["status"] = np.where(
    large_orders["order_id"].str.startswith("C")
    | (large_orders["quantity"] < 0),
    "取消/退货",
    "完成",
)
print("UCI Online Retail 公开数据：")
print(f"  {len(large_orders):,} 行 × {large_orders.shape[1]} 列")
print(
    "内存占用：",
    f"{large_orders.memory_usage(deep=True).sum() / 1024**2:.1f} MB",
)
large_orders.head()


In [ ]:
features = large_orders.assign(
    order_date=large_orders["order_time"].dt.date,
    month=large_orders["order_time"].dt.to_period("M").astype("string"),
    weekday=large_orders["order_time"].dt.day_name(),
    hour=large_orders["order_time"].dt.hour,
    is_weekend=large_orders["order_time"].dt.dayofweek >= 5,
    description_clean=large_orders["description"].str.strip().str.title(),
    order_label=large_orders["country"]
    .astype("string")
    .str.cat(large_orders["stock_code"], sep=" / "),
)
display(
    features[
        [
            "order_time",
            "month",
            "weekday",
            "hour",
            "is_weekend",
            "description_clean",
            "order_label",
        ]
    ].head()
)
print("月份跨度：", features["month"].min(), "至", features["month"].max())


## 27.8 独立迁移练习

替换一个字段或分组口径，并核对处理前后的行数与粒度。

先在下面单元格完成自己的版本；需要参考时再回看紧邻的示例或参考实现。


In [ ]:
# TODO: 在此粘贴或改写最接近的示例。
# 记录：我改了什么？预期会发生什么？实际观察到什么？
change_note = "待填写"
expected_change = "待填写"
observed_change = "运行后填写"
print({"修改": change_note, "预期": expected_change, "观察": observed_change})


## 27.9 本章实训：分组汇总与粒度

这一组实验专门训练“观察一个结果 → 只改一个变量 → 解释变化”。先运行第一个代码单元格，再运行第二个。


In [ ]:
import pandas as pd

orders = pd.DataFrame(
    {
        "region": ["华东", "华东", "华南", "华南"],
        "channel": ["线上", "线下", "线上", "线下"],
        "sales": [120, 80, 150, 100],
    }
)
summary = orders.groupby("region", as_index=False)["sales"].sum()
print(summary)
print("汇总表每一行代表一个地区")


### 27.9.1 第一个结果怎么读

先确认明细表一行代表一笔订单，再确认汇总表一行代表一个地区。`groupby` 的字段决定结果的粒度。

请记录：输入是什么、输出是什么、输出支持了哪一个结论。



In [ ]:
orders["sales_level"] = orders["sales"].map(
    lambda value: "高" if value >= 120 else "普通"
)
print(orders)
print(orders["sales_level"].value_counts())


### 27.9.2 第二个结果怎么读

第二个实验只增加一个分类列，不改变原始销售额。练习解释：什么时候应该新增列，什么时候应该直接筛选行？

迁移任务：把一个输入值、一个字段或一个图表参数换成自己的例子，再用一句话解释变化。



## 27.10 错误恢复：脏数据转换怎么办

真实数据和真实代码都会出问题。本节先观察问题，再用一个明确的检查或修复步骤恢复运行。


In [ ]:
import pandas as pd

raw = pd.Series(["12", "unknown", "18", ""])
converted = pd.to_numeric(raw, errors="coerce")
print("转换结果：")
print(converted)
print("无法转换的数量：", converted.isna().sum())
print("后续可以选择删除、填充或回查原始值。")


### 27.10.1 错误恢复步骤

1. 先看错误类型、字段或数据形状。
2. 判断问题发生在输入、处理中间结果还是输出。
3. 修复后重新检查结果，而不是只让代码不报错。

errors="coerce" 会把无法转换的值记录为缺失，适合先完成质量盘点；不要在没有统计数量前直接删除。

迁移任务：把示例中的输入换成一组会触发问题的数据，并记录你的修复规则。



## 27.11 易错点提醒

- 直接对object列使用日期运算
- 正则提取失败后不检查缺失
- 使用结果变量构造导致数据泄漏的特征


## 27.12 练习与作业

1. 清洗手机号中的空格和连字符
2. 解析注册日期
3. 构造注册月份和账户天数

提交前检查：代码可从上到下运行，关键中间结果可核对，结论注明计算口径。

## 27.13 练习路径

1. **跟练**：先运行示例，确认输出结构，再完成“清洗手机号中的空格和连字符”。
2. **独立完成**：不复制示例代码，完成“解析注册日期”，并保留一个中间结果用于检查。
3. **迁移挑战**：尝试“构造注册月份和账户天数”，用一两句话说明你修改了什么。

### 27.13.1 完成标准

- 代码从上到下运行不报错，关键变量类型和形状符合预期。
- 至少输出一个可核对的数值、表格或图形，并写明计算口径。
- 结论能够回答任务问题，同时说明一个限制或未验证的假设。

### 27.13.2 分级提示

- **提示 1**：先复用示例中的数据结构和变量命名。
- **提示 2**：把任务拆成“准备数据 → 计算 → 检查 → 表达”四步。
- **提示 3**：运行隐藏答案前，先用 type()、shape、head() 或断言定位问题。


In [ ]:
import pandas as pd

# TODO: 清洗手机号，去除所有非数字字符
# TODO: 解析注册日期
# TODO: 提取注册月份
# TODO: 计算账户天数（以2026-04-01为参考日期）
# TODO：请在下方完成 —— 23.13 练习与作业 1. 清洗手机号中的空格和连字符 2. 解析注册日期 3. 构造注册月份和账户天数 提交前检查：


In [ ]:
import pandas as pd

users = pd.DataFrame(
    {
        "phone": ["138-0000-1234", " 139 0000 5678 "],
        "registered_at": ["2025-12-15", "2026-02-08"],
    }
)
users["phone_clean"] = users["phone"].str.replace(r"\D", "", regex=True)
users["registered_at"] = pd.to_datetime(users["registered_at"])
users["register_month"] = (
    users["registered_at"].dt.to_period("M").astype("string")
)
users["account_days"] = (
    pd.Timestamp("2026-04-01") - users["registered_at"]
).dt.days
print(users)


## 27.14 小结

结合字符串、日期和数值列构造可分析的业务特征。

**迁移思考**：

1. 如果需要提取用户邮箱的用户名部分（@符号之前），正则表达式应该如何写？
2. 为什么特征构造要避免使用未来信息？举一个会导致数据泄漏的例子。



### 27.14.1 你已经掌握

- 批量清洗文本列
- 解析和拆解日期
- 计算时间差
- 构造分类与数值特征



### 27.14.2 验收标准

- 输入、计算和输出单元格完整。
- 关键变量类型、形状或数值可核对。
- 结论引用输出证据，并注明适用范围。



### 27.14.3 需要注意

- 直接对object列使用日期运算
- 正则提取失败后不检查缺失
- 使用结果变量构造导致数据泄漏的特征



### 27.14.4 完成检查

- [ ] 能够批量清洗文本列
- [ ] 能够解析和拆解日期
- [ ] 能够计算时间差
- [ ] 能够构造分类与数值特征



### 27.14.5 排错顺序

1. 从上到下重新运行依赖单元格。
2. 检查变量类型、列名、形状和缺失值。
3. 缩小输入范围，定位产生错误的最小步骤。
4. 修复后重新运行完整流程。

